# 医学实体关系抽取训练笔记本(CMeIE 数据集) - 完整修复版

**目标**:训练一个能抽取中文医学三元组(实体, 关系, 实体)的 GPLinker 模型。

**数据**:CMeIE 中文医学信息抽取数据集(HuggingFace 直接加载,43 种医学关系)

**本版已内置全部修复**(踩坑总结):
1. ✅ AdamW 导入(从 torch.optim,与版本无关)
2. ✅ tokenizer 路径(vocab.txt → 目录)
3. ✅ 启用训练 + 跳过 test(首次训练无模型)
4. ✅ CMeIE 数据格式(subject 保留,不切掉 `@` 前缀)
5. ✅ 演示文本换成医学文本(不再打印娱乐文本)
6. ✅ 固定目录,不嵌套

**直接从头运行到尾即可,无需任何手动修改。**

## 第 0 步:固定目录 + 检查 GPU

In [ ]:
import os
os.chdir('/content')
print('工作目录:', os.getcwd())

import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 第 1 步:克隆项目 + 安装依赖

In [ ]:
import shutil, os, subprocess
PROJ = '/content/gplinker-med'
if os.path.exists(PROJ):
    shutil.rmtree(PROJ)
os.makedirs(PROJ)
os.chdir(PROJ)
subprocess.run("git clone --depth 1 https://github.com/taishan1994/pytorch_GlobalPointer_triple_extraction.git .", shell=True)
print('克隆完成,当前目录:', os.getcwd())

In [ ]:
import subprocess
subprocess.run("pip install torchcrf tensorboardX datasets -q", shell=True)
import transformers
print('transformers:', transformers.__version__)
print('依赖就绪')

## 第 2 步:修复 main.py 的 3 处问题(已内置)

1. `AdamW` 导入 → torch.optim
2. tokenizer 路径 → 目录
3. 启用训练 + 跳过 test

另外把 main.py 里的演示文本(娱乐类)换成医学文本。

In [ ]:
# 修复1: AdamW 导入
with open('utils/train_utils.py', encoding='utf-8') as f:
    c = f.read()
c = c.replace('from transformers import AdamW, get_linear_schedule_with_warmup',
              'from torch.optim import AdamW\nfrom transformers import get_linear_schedule_with_warmup')
c = c.replace('from transformers import AdamW', 'from torch.optim import AdamW')
with open('utils/train_utils.py', 'w', encoding='utf-8') as f:
    f.write(c)
print('修复1: AdamW ✓')

# 修复2: tokenizer 路径
with open('main.py', encoding='utf-8') as f:
    c = f.read()
c = c.replace("BertTokenizer.from_pretrained('model_hub/chinese-bert-wwm-ext/vocab.txt')",
              "BertTokenizer.from_pretrained('model_hub/chinese-bert-wwm-ext')")
with open('main.py', 'w', encoding='utf-8') as f:
    f.write(c)
print('修复2: tokenizer 路径 ✓')

# 修复3: 启用训练 + 跳过 test
with open('main.py', encoding='utf-8') as f:
    c = f.read()
c = c.replace('# bertForNer.train()', 'bertForNer.train()')
c = c.replace('        bertForNer.test(model_path)', '        # bertForNer.test(model_path)')
with open('main.py', 'w', encoding='utf-8') as f:
    f.write(c)
print('修复3: 启用训练 + 跳过 test ✓')

# 修复4: 演示文本换成医学文本
with open('main.py', encoding='utf-8') as f:
    c = f.read()
start = c.find('texts = [')
end = c.find(']', start)
medical_texts = "texts = [\n" + \
    "        '氟西汀是治疗抑郁症的常用药物,禁与单胺氧化酶抑制剂合用',\n" + \
    "        '抑郁障碍的核心症状包括情绪低落和兴趣丧失',\n" + \
    "        '急性胰腺炎需要进行CT检查来明确诊断',\n" + \
    "        '药物治疗是高血压的主要治疗方式,包括钙离子拮抗剂',\n" + \
    "        ]"
c = c[:start] + medical_texts + c[end+1:]
with open('main.py', 'w', encoding='utf-8') as f:
    f.write(c)
print('修复4: 演示文本换为医学 ✓')
print('\n全部修复完成,无需手动改代码')

## 第 3 步:准备医学数据(CMeIE)

**关键修复**:CMeIE 的 text 是 `主题疾病@正文` 格式,subject 来自 `@` 前缀。**保留完整文本,不切分**,否则 subject 找不到。

In [ ]:
import json, os, random
from datasets import load_dataset

ds = load_dataset('Aunderline/CMeIE', split='train')
print('CMeIE 加载成功, 条数:', len(ds))
print('示例:', ds[0])

os.makedirs('data/ske/raw_data', exist_ok=True)
os.makedirs('data/ske/mid_data', exist_ok=True)

def build_medical_subset(dataset, path, n, seed=42):
    """CMeIE → 训练/验证集。保留完整 text(含 @ 前缀的 subject),object 取 @value"""
    random.seed(seed)
    idx = random.sample(range(len(dataset)), min(n, len(dataset)))
    preds = []
    cnt = 0
    with open(path, 'w', encoding='utf-8') as f:
        for i in idx:
            item = dataset[i]
            text = item['text']  # 保留完整文本(不切分@前缀)
            spo = item.get('spo_list', [])
            new_spo = []
            for s in spo:
                obj = s.get('object')
                if isinstance(obj, dict):
                    obj = obj.get('@value', '')
                if isinstance(obj, str) and obj:
                    new_spo.append({'subject': s['subject'], 'predicate': s['predicate'], 'object': obj})
                    if s['predicate'] not in preds:
                        preds.append(s['predicate'])
            if new_spo:
                f.write(json.dumps({'text': text, 'spo_list': new_spo}, ensure_ascii=False) + '\n')
                cnt += 1
    return cnt, preds

train_cnt, train_preds = build_medical_subset(ds, 'data/ske/raw_data/train_data.json', 2000)
dev_cnt, _ = build_medical_subset(ds, 'data/ske/raw_data/dev_data.json', 200, seed=7)

with open('data/ske/mid_data/predicates.json', 'w', encoding='utf-8') as f:
    json.dump(train_preds, f, ensure_ascii=False, indent=2)

print(f'训练集: {train_cnt} 条, 关系数: {len(train_preds)}')
print(f'验证集: {dev_cnt} 条')
print('\n医学关系列表:', train_preds)

In [ ]:
# 验证数据质量:subject 是否都在文本里(确保标签不丢失)
with open('data/ske/raw_data/train_data.json', encoding='utf-8') as f:
    lines = f.readlines()
hit = total = 0
for line in lines[:200]:
    item = json.loads(line)
    text = item['text']
    for spo in item['spo_list']:
        total += 1
        if spo['subject'] in text:
            hit += 1
print(f'subject 命中率: {hit}/{total} = {hit/total:.0%}')
if hit/total < 0.8:
    print('警告: 命中率偏低,可能影响训练')
else:
    print('数据质量 OK')

## 第 4 步:下载预训练模型

In [ ]:
from transformers import AutoTokenizer, AutoModel
save_dir = 'model_hub/chinese-bert-wwm-ext'
os.makedirs(save_dir, exist_ok=True)
tok = AutoTokenizer.from_pretrained('hfl/chinese-bert-wwm-ext')
tok.save_pretrained(save_dir)
model = AutoModel.from_pretrained('hfl/chinese-bert-wwm-ext')
model.save_pretrained(save_dir)
print('模型已保存:', os.listdir(save_dir))

## 第 5 步:训练

num_tags 自动读取。训练 3 epochs,结束后自动保存模型 + 用医学演示文本测试。

In [ ]:
import subprocess, json

with open('data/ske/mid_data/predicates.json', encoding='utf-8') as f:
    num_tags = len(json.load(f))
print('num_tags =', num_tags)

train_cmd = f"python main.py " + \
    '--bert_dir="model_hub/chinese-bert-wwm-ext/" ' + \
    '--data_dir="./data/ske/" ' + \
    '--log_dir="./logs/" ' + \
    '--output_dir="./checkpoints/" ' + \
    f'--num_tags={num_tags} ' + \
    '--seed=123 --max_seq_len=128 --lr=3e-5 --other_lr=1e-4 ' + \
    '--train_batch_size=8 --train_epochs=3 --eval_batch_size=8 ' + \
    '--use_tensorboard="False"'

result = subprocess.run(train_cmd, shell=True, capture_output=True, text=True)
print('=== 标准输出(尾部) ===')
print(result.stdout[-2500:])
print('=== 错误输出(尾部) ===')
print(result.stderr[-2500:])

## 第 6 步:用医学文本测试

加载训练好的模型,测试医学抽取效果。

In [ ]:
import torch, json
from types import SimpleNamespace
from transformers import BertTokenizer
from model import GlobalPointerRe
from utils.train_utils import load_model_and_parallel
import importlib.util, sys

args = SimpleNamespace(
    output_dir='./checkpoints/', bert_dir='model_hub/chinese-bert-wwm-ext/',
    data_dir='./data/ske/', log_dir='./logs/', num_tags=0, seed=123,
    gpu_ids='0', max_seq_len=128, eval_batch_size=8, train_epochs=3,
    dropout_prob=0.1, lr=3e-5, other_lr=1e-4, max_grad_norm=1,
    use_tensorboard='False', warmup_proportion=0.1, weight_decay=0.01,
    adam_epsilon=1e-8, train_batch_size=8, use_dev_num=32, eval_steps=32,
)
with open('data/ske/mid_data/predicates.json', encoding='utf-8') as f:
    predicates = json.load(f)
args.num_tags = len(predicates)
id2tag = {i: p for i, p in enumerate(predicates)}

tokenizer = BertTokenizer.from_pretrained('model_hub/chinese-bert-wwm-ext')
model = GlobalPointerRe(args)
model, device = load_model_and_parallel(model, args.gpu_ids, './checkpoints/bert/model.pt')
model.eval()
print('模型加载成功, 关系数:', args.num_tags)

sys.argv = ['main.py', '--num_tags=1']
spec = importlib.util.spec_from_file_location('main_mod', 'main.py')
main_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(main_mod)
bertForRe = main_mod.BertForRe(args, None, None, None, id2tag, None, model, device)

texts = [
    "氟西汀是治疗抑郁症的常用药物,禁与单胺氧化酶抑制剂合用",
    "抑郁障碍的核心症状包括情绪低落和兴趣丧失",
    "急性胰腺炎需要进行CT检查来明确诊断",
]
for text in texts:
    print("=" * 60)
    print("文本:", text)
    bertForRe.predict(text, model, tokenizer)